# CSV a SQL

En este archivo vamos a crear inserts de SQL a partir de datos de un CSV.

Vamos a seguir varios pasos:

+ Instalar dependencias nuevas.
+ Llenar tablas fuertes
+ Llenar tablas débiles

Ejemplo:

Voy a declarar una variable:

In [1]:
x = 5

Le voy a sumar 5 a x

In [17]:
x = x + 5

Voy a ver el valor de x: 

In [18]:
x

30

Importación de librerías

In [1]:
import pandas as pd

Ruta global y lectura del dataset

In [2]:
BASE_PATH = ''
ds_players = pd.read_csv('all_players.csv', sep=',')
male_players = pd.read_csv('male_players.csv', sep=',')
female_players = pd.read_csv('female_players.csv', sep=',')

/tmp/ipykernel_90188/820602085.py:2: DtypeWarning: Columns (33: Standing Tackle) have mixed types. Specify dtype option on import or set low_memory=False.
  ds_players = pd.read_csv('all_players.csv', sep=',')


Ejemplo para listar columnas por nombre

In [3]:
male_players[['Name', 'Age']]

,Name,Age
0,Kylian Mbappé,25
1,Rodri,28
2,Erling Haaland,24
3,Jude Bellingham,21
4,Vini Jr.,24
...,...,...
16156,Engson Singh,21
16157,Ousmane Sylla,23
16158,Wei Feng,22
16159,Won Hui Do,19


Convertir la altura (182cm / 6'0") a 182

In [24]:
# x = '1::2::3'
# y = x.split('::') -> ['1','2','3']
alturas = ds_players['Height'] # list
for altura in alturas:
  # 182cm / 6'0"
  tmp = altura.split('cm')
  estatura = int(tmp[0])
  print(estatura/100, 'm')

1.82 m
1.91 m
1.95 m
1.86 m
1.76 m
1.81 m
1.88 m
1.78 m
1.96 m
1.93 m
2.0 m
1.74 m
1.93 m
1.87 m
1.75 m
1.71 m
1.69 m
1.76 m
1.87 m
1.85 m
1.82 m
1.88 m
1.73 m
1.88 m
1.9 m
1.77 m
1.95 m
1.93 m
1.86 m
1.84 m
1.78 m
1.77 m
1.84 m
1.75 m
1.85 m
1.81 m
1.91 m
1.79 m
1.83 m
1.83 m
1.9 m
1.8 m
1.95 m
1.72 m
1.83 m
1.87 m
1.9 m
1.86 m
1.74 m
1.82 m
1.88 m
1.71 m
1.72 m
1.76 m
1.73 m
1.78 m
1.8 m
1.74 m
1.95 m
1.85 m
1.87 m
1.77 m
1.9 m
1.93 m
1.78 m
1.8 m
1.88 m
1.85 m
1.86 m
1.82 m
1.92 m
1.68 m
1.8 m
1.83 m
1.72 m
1.75 m
1.79 m
1.99 m
1.89 m
1.92 m
1.77 m
1.9 m
1.89 m
1.78 m
1.83 m
1.91 m
1.88 m
1.88 m
1.78 m
1.81 m
1.8 m
1.81 m
1.92 m
1.87 m
1.8 m
1.7 m
1.76 m
1.95 m
1.75 m
1.81 m
1.8 m
1.89 m
1.8 m
1.76 m
1.8 m
1.74 m
1.89 m
1.72 m
1.73 m
1.8 m
1.87 m
1.87 m
1.81 m
1.92 m
1.86 m
1.74 m
1.78 m
1.92 m
1.8 m
1.77 m
1.79 m
1.78 m
1.86 m
1.92 m
1.85 m
1.78 m
1.85 m
1.94 m
1.9 m
1.83 m
1.85 m
1.72 m
1.86 m
1.75 m
1.83 m
1.82 m
1.81 m
1.87 m
1.89 m
1.8 m
1.7 m
1.88 m
1.72 m
1.89 m
1.85 m
1.93 m

Listar los pies preferidos de los futbolistas sin repeticiones

In [4]:
pies_con_repeticiones = male_players['Preferred foot']
pies = list(pies_con_repeticiones.unique())
pies

['Right', 'Left']

Listar las nacionalidades de los futbolistas sin repeticiones

```sql
CREATE TABLE nations (
    id INT NOT NULL AUTO_INCREMENT,
    name VARCHAR(40) NOT NULL,
    PRIMARY KEY (id)
);
```

In [15]:
nacionalidades = list(ds_players['Nation'].unique())
nacionalidades = sorted(nacionalidades)
text = '-- migrate:up \n\n'
id = 1
for n in nacionalidades:
  nuevo = n.replace("'", "''")
  tmp = f"INSERT INTO nations (id, name) VALUES ({id}, '{nuevo}');\n"
  text = text + tmp
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM nations;'
with open('inserts_nations.sql', 'w') as archivo:
  archivo.write(text)

Insert de ligas

```sql
CREATE TABLE leagues (
    id INT NOT NULL AUTO_INCREMENT,
    name VARCHAR(40) NOT NULL,
    nation_id INT NOT NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_leagues_nation
        FOREIGN KEY (nation_id) REFERENCES nations(id)
);
```

In [21]:
ligas = list(ds_players['League'].unique())
ligas = sorted(ligas)
text = '-- migrate:up \n\n'
ligas_dicc = {}
id = 1
for n in ligas:
  nuevo = n.replace("'", "''")
  tmp = f"INSERT INTO leagues (id, name, nation_id) VALUES ({id}, '{nuevo}', 1);\n"
  ligas_dicc[nuevo] = id
  text = text + tmp
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM leagues;'
with open('inserts_leagues.sql', 'w') as archivo:
  archivo.write(text)

ligas_dicc

{'1A Pro League': 1,
 '3. Liga': 2,
 '3F Superliga': 3,
 'A-League': 4,
 'Allsvenskan': 5,
 'Arkema PL': 6,
 'Barclays WSL': 7,
 'Bundesliga': 8,
 'Bundesliga 2': 9,
 'CSL': 10,
 'CSSL': 11,
 'Calcio A Femminile': 12,
 'Ceska Liga Žen': 13,
 'EFL Championship': 14,
 'EFL League One': 15,
 'EFL League Two': 16,
 'Eliteserien': 17,
 'Eredivisie': 18,
 'Finnliiga': 19,
 'GPFBL': 20,
 'Hellas Liga': 21,
 'ISL': 22,
 'K League 1': 23,
 'LALIGA EA SPORTS': 24,
 'LALIGA HYPERMOTION': 25,
 'Libertadores': 26,
 'Liga Azerbaijan': 27,
 'Liga Colombia': 28,
 'Liga Cyprus': 29,
 'Liga F': 30,
 'Liga Hrvatska': 31,
 'Liga Portugal': 32,
 'Liga Portugal Feminino': 33,
 "Ligue 1 McDonald''s": 34,
 'Ligue 2 BKT': 35,
 'MLS': 36,
 'Magyar Liga': 37,
 'NWSL': 38,
 'Nederland Vrouwen Liga': 39,
 'PKO BP Ekstraklasa': 40,
 'Premier League': 41,
 'Primera División': 42,
 'ROSHN Saudi League': 43,
 'SSE Airtricity PD': 44,
 'SUPERLIGA': 45,
 'Schweizer Damen Liga': 46,
 'Scottish Prem': 47,
 "Scottish Women

Insert de equipos

```sql
CREATE TABLE teams (
    id INT NOT NULL AUTO_INCREMENT,
    name VARCHAR(10) NOT NULL,
    league_id INT NOT NULL,
    PRIMARY KEY (id),
    CONSTRAINT fk_teams_league
        FOREIGN KEY (league_id) REFERENCES leagues(id)
);
```

In [23]:
lista = ds_players[['Team', 'League']]
equipos = []

text = '-- migrate:up \n\n'
id = 1
for row in lista.itertuples(index=False):
  team = row.Team.replace("'", "''")
  league = row.League.replace("'", "''")

  if team not in equipos:
    league_id = ligas_dicc[league]
    
    tmp = f"INSERT INTO teams (id, name, league_id) VALUES ({id}, '{team}', {league_id});\n"
    text = text + tmp
    id = id + 1

    equipos.append(team)
text = text + '\n-- migrate:down \n\nDELETE FROM teams;'
with open('inserts_teams.sql', 'w') as archivo:
  archivo.write(text)
